---
### __1.1 Loading__
---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
caminho_arquivo = '../data/raw/MiningProcess_Flotation_Plant_Database.csv'

df = pd.read_csv(caminho_arquivo, sep=',', decimal=',', parse_dates=["date"])

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.head(10)

---
### __1.2 Temporal structure__
---

In [ ]:
# Records per hour
rows_per_hour = df.groupby('date').size()
print(rows_per_hour.value_counts().sort_index())
print(rows_per_hour[rows_per_hour != 180])

In [ ]:
# Missing hours
full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="h")

missing = full_range.difference(rows_per_hour.index)

print(f"expected: {len(full_range)}, present: {len(rows_per_hour)}, missing: {len(missing)}")

missing_s = missing.to_series()
block_id = missing_s.diff().ne(pd.Timedelta(hours=1)).cumsum()
blocks = missing_s.groupby(block_id.values).agg(["min", "max", "count"])
print(blocks)


In [ ]:
# Column that vary

max_distinct_per_hour = df.groupby("date").nunique().max().sort_values()
print(max_distinct_per_hour)

In [ ]:
target = "% Silica Concentrate"
distinct_target = df.groupby("date")[target].nunique()
varying = distinct_target[distinct_target > 1]

print(f"hours with varying target: {len(varying)}")
print(f"from {varying.index.min()} to {varying.index.max()}")

In [ ]:
cols = ['% Iron Concentrate', '% Iron Feed', '% Silica Feed', '% Silica Concentrate', 'Flotation Column 01 Level', 'Flotation Column 03 Air Flow']
mask = (df['date'] >= '2017-04-11 01:00:00') & (df['date'] < '2017-04-21 02:00:00')

s_1 = df.loc[mask]
#diffs = s_1[cols].diff()
diffs = s_1.diff()

print((diffs != 0).sum())           # quantas vezes cada coluna mudou na hora


In [ ]:
diffs.abs().describe()       # tamanho das variações

In [ ]:
step = df.groupby("date")[target].diff()
step_std = step.groupby(df["date"]).std()

print(step_std.loc[varying.index].describe())

In [ ]:
hourly = df.groupby("date")[target].agg(["first", "last"])
next_first = hourly["first"].shift(-1, freq="h").reindex(hourly.index)
gap_to_next = (hourly["last"] - next_first).abs()

print("varying hours:")
print(gap_to_next.loc[varying.index].describe())
print("constant hours (reference):")
print(gap_to_next.drop(varying.index).describe())

In [ ]:

sample_hours = varying.index[[0, len(varying) // 2, -1]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, hour in zip(axes, sample_hours):
    window = df[df["date"].between(hour - pd.Timedelta(hours=1), hour + pd.Timedelta(hours=1))]
    ax.plot(window[target].to_numpy())
    ax.set_title(str(hour))
    ax.set_xlabel("row (20 s each)")
    ax.set_ylabel(target)
plt.tight_layout()

### Finding: interpolated target values (data leakage risk)

The lab assay for `% Silica Concentrate` is reported once per hour, so within an hour
all 180 rows (20 s each) should carry the same value. In **310 hours (~7.5%)**, spread
from 2017-03-12 to 2017-09-06, the target changes on every row instead.

**Evidence that these values are linear interpolations toward the next hour:**

| Check | Interpolated hours (n = 310) | Regular hours (n = 3,785) |
|---|---|---|
| Std. dev. of row-to-row steps within the hour | max $5.0 \times 10^{-11}$ (constant step) | — |
| \|last value of hour *h* − first value of hour *h*+1\| (median) | 0.0008 | 0.23 |

- **Shape:** the step between consecutive rows is constant within each hour, i.e.
  the values lie on a straight line.
- **Destination:** each ramp ends at the value of the next hour, roughly 300× closer
  than the typical hour-to-hour jump between real assays.
- **Blocks:** interpolated hours appear in consecutive runs (e.g. a single straight
  ramp spanning three hours around 2017-05-16 19:00), consistent with missing lab
  results being filled by connecting the last known assay to the next one.

**Implication:** in these hours the target is not a measurement. It was built using
the *next* lab result, which did not exist at prediction time. Training or evaluating
on these rows leaks future information, and chronological splits do not catch it
because the leak is inside each row. How to handle them (exclude or flag) is decided
in section 1.6, together with the target definition.

In [ ]:
feed = "% Iron Feed"

hourly_feed = df.groupby("date")[feed].first().asfreq("h")
block_id = hourly_feed.ne(hourly_feed.shift()).cumsum()


runs = hourly_feed.groupby(block_id.values).agg(
    start=lambda s: s.index.min(),
    hours="size",
    value="first",
)
runs = runs[runs["value"].notna()]

print(f"blocks: {len(runs)}")
print(runs["hours"].describe())
print(runs["hours"].value_counts().sort_index().head(15))


------------------------
__Finding__
- To be analysed in next phases as a feature: “hours since last feed updating"
------------------------

In [ ]:
print(runs.loc[runs["hours"].isin([6, 9]), "start"].dt.hour.value_counts().sort_index())
print(runs[runs["hours"] > 48])

---
__Finding__

- Standard time feed: 01:00/ 07:00/ 16:00
- Same feed value(64.03) per 33 days: between 2017-05-13 01:00:00 - 2017-06-14 00:00:00
---

---
### __1.3 Data quality__
---

In [ ]:
# Check Nulls

nulls = df.isna().sum()
print(nulls[nulls > 0] if nulls.any() else "no nulls")

In [ ]:
# Check duplicated

print(f"extra copies: {df.duplicated().sum()}")
print(f"rows involved in duplicates: {df.duplicated(keep=False).sum()}")

consecutive = df.eq(df.shift()).all(axis=1)
print(f"rows identical to the previous row: {consecutive.sum()}")

print(df.loc[df.duplicated(), "date"].dt.to_period("M").value_counts().sort_index())

In [ ]:
# Validate business rules x data

range_rules = {
    # assays (%)
    "% Iron Feed": (0, 100),
    "% Silica Feed": (0, 100),
    "% Iron Concentrate": (0, 70),  # hematite (Fe2O3) is ~69.9% Fe
    "% Silica Concentrate": (0, 100),
    # reagents and pulp
    "Starch Flow": (0, None),  # flows cannot be negative
    "Amina Flow": (0, None),
    "Ore Pulp Flow": (0, None),
    "Ore Pulp pH": (8, 11),  # reverse cationic flotation runs ~10
    "Ore Pulp Density": (1.0, 3.0),  # pulp denser than water, below solids density
}
for i in range(1, 8):
    range_rules[f"Flotation Column 0{i} Air Flow"] = (0, None)
    range_rules[f"Flotation Column 0{i} Level"] = (0, None)


def count_range_violations(frame: pd.DataFrame, rules: dict) -> pd.DataFrame:
    result = {}
    for col, (low, high) in rules.items():
        values = frame[col]
        bad = pd.Series(False, index=values.index)
        if low is not None:
            bad |= values < low
        if high is not None:
            bad |= values > high
        result[col] = {
            "low": low,
            "high": high,
            "violations": int(bad.sum()),
            "pct": round(100 * bad.mean(), 3),
        }
    return pd.DataFrame(result).T


print(count_range_violations(df, range_rules))

cross_rules = {
    "feed: Fe + SiO2 > 100": df["% Iron Feed"] + df["% Silica Feed"] > 100,
    "concentrate: Fe + SiO2 > 100": df["% Iron Concentrate"] + df["% Silica Concentrate"] > 100,
}
print(pd.Series({name: int(mask.sum()) for name, mask in cross_rules.items()}, name="violations"))

In [ ]:
# Check sensor lock

fast_cols = [c for c in df.columns if c not in ("date", "% Iron Feed", "% Silica Feed")]


def longest_run(values: pd.Series) -> int:
    block_id = values.ne(values.shift()).cumsum()
    return int(values.groupby(block_id).size().max())


stuck = pd.DataFrame({"longest_run_rows": {c: longest_run(df[c]) for c in fast_cols}})
stuck["longest_run_hours"] = (stuck["longest_run_rows"] / 180).round(1)
print(stuck.sort_values("longest_run_rows", ascending=False))

In [ ]:
# Search event in the time

def stuck_runs(frame: pd.DataFrame, col: str, min_rows: int) -> pd.DataFrame:
    values = frame[col]
    block_id = values.ne(values.shift()).cumsum()
    runs = frame.groupby(block_id.values).agg(
        start=("date", "min"),
        end=("date", "max"),
        rows=("date", "size"),
    )
    runs = runs[runs["rows"] >= min_rows].reset_index(drop=True)
    runs.insert(0, "column", col)
    return runs


fast_process_cols = [c for c in fast_cols if c not in ("% Iron Concentrate", "% Silica Concentrate")]

events = pd.concat(
    [stuck_runs(df, c, min_rows=360) for c in fast_process_cols]
    + [stuck_runs(df, c, min_rows=180 * 24) for c in ("% Iron Concentrate", "% Silica Concentrate")],
    ignore_index=True,
)
events["hours"] = (events["rows"] / 180).round(1)
print(events.sort_values(["start", "column"]).to_string())

---
### Finding: plausible but false values

No nulls and no physically impossible values were found: all range rules (assays
between 0 and 100%, non-negative flows, pH 8–11, pulp density 1–3) and cross-column
rules (Fe + SiO₂ ≤ 100%) pass. The quality problems in this dataset are values that
look valid but were not measured.

**Frozen acquisition (whole plant).** On 2017-07-11 06:00–10:00 (3.7 h) all 19 process
variables hold the same value simultaneously. This accounts for most of the 1,171
duplicated rows, all of which are consecutive and fall almost entirely in June–July.

**Frozen sensor groups.** Subsets of sensors freeze together, suggesting a shared
network or I/O card:

| Period | Sensors | Duration |
|---|---|---|
| 2017-03-10 → 2017-04-11 11:00 | Air flow, columns 04 and 05 | ~460 h (from dataset start) |
| 2017-06-21 14:00–17:00 | Level 01, air flow 06 and 07, pH | 2.5 h |
| 2017-07-31 11:00–16:00 | pH | 5.1 h |
| 2017-08-02 08:00–15:00 | Air flow 01, 02, 03, 06, 07, pH | 7.7 h |

**Repeated lab assays.** Both concentrate assays (Fe and SiO₂) stay constant for
28–74 consecutive hours in three periods (2017-03-31, 2017-07-31, 2017-08-17),
suggesting missing results were forward-filled. Unlike the interpolated hours
(section 1.2), this does not leak future information, but it creates labels that were
never measured.

**Implication:** each pattern becomes a data quality rule in the silver layer
(stage 2): flag frozen runs per sensor, flag whole-row freezes, and flag stale lab
assays.
---

---
### __1.4 Univariate analysis__
---

In [ ]:
# Statistical summary with skewness and outliers

num_cols = df.columns.drop("date")

q1 = df[num_cols].quantile(0.25)
q3 = df[num_cols].quantile(0.75)
iqr = q3 - q1
outside = (df[num_cols] < q1 - 1.5 * iqr) | (df[num_cols] > q3 + 1.5 * iqr)

summary = df[num_cols].describe().T
summary["skew"] = df[num_cols].skew()
summary["outliers_pct"] = (100 * outside.mean()).round(2)
print(summary.round(3).to_string())

In [ ]:
#Histograms

fig, axes = plt.subplots(6, 4, figsize=(18, 20))
for ax, col in zip(axes.flat, num_cols):
    ax.hist(df[col], bins=60)
    ax.set_title(col, fontsize=9)
for ax in axes.flat[len(num_cols):]:
    ax.set_visible(False)
plt.tight_layout()

In [ ]:
# Variables in  the time

hourly = df.groupby("date")[num_cols].mean().asfreq("h")

fig, axes = plt.subplots(len(num_cols), 1, figsize=(16, 2 * len(num_cols)), sharex=True)
for ax, col in zip(axes, num_cols):
    ax.plot(hourly.index, hourly[col], linewidth=0.6)
    ax.set_ylabel(col, rotation=0, ha="right", fontsize=8)
plt.tight_layout()

In [ ]:
# Target distribution

lab = df.groupby("date")[target].first().drop(varying.index)
print(lab.describe())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lab, bins=50)
ax.set_xlabel(target)
ax.set_title("Hourly lab assays (interpolated hours excluded)")


In [ ]:
# Check variables at the limit

at_min = df[num_cols].eq(df[num_cols].min())
at_max = df[num_cols].eq(df[num_cols].max())

bounds = pd.DataFrame({
    "min": df[num_cols].min(),
    "pct_at_min": (100 * at_min.mean()).round(2),
    "max": df[num_cols].max(),
    "pct_at_max": (100 * at_max.mean()).round(2),
})
print(bounds.sort_values("pct_at_min", ascending=False).to_string())

In [ ]:
# linear ramps in the input variables

def linear_ramps(frame: pd.DataFrame, col: str, min_rows: int, tol: float = 2e-3) -> pd.DataFrame:
    step = frame[col].diff()
    new_block = (step - step.shift()).abs().gt(tol) | step.abs().le(tol) | step.isna()
    block_id = new_block.cumsum()
    runs = frame.groupby(block_id.values).agg(
        start=("date", "min"),
        end=("date", "max"),
        rows=("date", "size"),
    )
    runs = runs[runs["rows"] >= min_rows].reset_index(drop=True)
    runs.insert(0, "column", col)
    return runs


ramps = pd.concat([linear_ramps(df, c, min_rows=180) for c in fast_process_cols], ignore_index=True)
ramps["hours"] = (ramps["rows"] / 180).round(1)
print(ramps.sort_values(["start", "column"]).to_string())

In [ ]:
mn, mx = df[num_cols].min(), df[num_cols].max()
band = 0.01 * (mx - mn)

near = pd.DataFrame({
    "pct_near_min": (100 * df[num_cols].le(mn + band).mean()).round(2),
    "pct_near_max": (100 * df[num_cols].ge(mx - band).mean()).round(2),
})
print(near.sort_values("pct_near_min", ascending=False).head(10).to_string())

---
### Findings: univariate analysis

**Target (hourly lab assays, interpolated hours excluded, n = 3,787).** Mean 2.19%,
median 1.90%, standard deviation **1.03 pp**, right-skewed (most values between 1% and
2.5%, tail up to 5.5%), with almost nothing below 1%. The standard deviation is the
reference for model error: always predicting the mean already gives an RMSE of about
1.03 pp.

**Outliers must be judged over time, not on the histogram.** The IQR rule flags 14.8%
of `Ore Pulp Flow` as outliers, but the time series shows a change of regime from
August on (oscillating between 380 and 418 instead of holding near 400), not
anomalous points.

**Regime changes (non-stationarity).** Starch flow steps up from ~2,000 to ~3,000+ in
early May; air flow in columns 01–03 moves from 250 to 300 in late April and columns
06–07 start using 350 in August; pulp flow changes behaviour in August; pH drops in
late August. A model trained on March–April never sees the later regimes, which is
why validation must be walk-forward (stage 5).

**Setpoint variables.** Air flows in columns 01, 02, 03, 06 and 07 concentrate on
250, 300 and 350, and levels peak at round values (400, 450, 500, 600): they are
operator-set, changing in steps.

**Values piling up near instrument limits.** No sensor repeats its exact minimum or
maximum (0.00% of rows), but some accumulate within 1% of their range: pulp density
near 1.52 (5.3% of rows), column 03 level near ~880 (2.4%), pH near 8.75 (1.5%).
This suggests sensors near the end of their range (censored values) or, for density,
real dilute-pulp operation. These rows are flagged, not removed.

**Linear interpolation also affects the inputs.** 639 linear ramps lasting at least
one hour were found in 18 process variables, about 2.8% of all sensor values (up to
10.2% of `Ore Pulp Flow`). The longest ramps occur in several sensors at once and
coincide with periods where the lab assay is repeated (e.g. 2017-08-17 → 2017-08-18,
2017-03-31 → 2017-04-01), pointing to data outages filled by interpolating inputs and
forward-filling assays. As with the target, interpolated inputs contain future
information: features computed over these periods would leak, so ramps must be
flagged in every variable in the silver layer.
---

---
### __1.5 Multivariate analysis__
---

In [ ]:
# Hourly Table
hourly = df.groupby("date")[num_cols].mean().asfreq("h")
hourly.loc[hourly.index.isin(varying.index), target] = np.nan

#print(hourly)
print(hourly.shape, hourly[target].notna().sum())

In [ ]:
# Correlation matrix

corr = hourly.corr(method="spearman")
print(corr[target].drop(target).sort_values(key=abs, ascending=False).round(3).to_string())

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr)), corr.index, fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()

In [ ]:
# Lagged correlation

def lagged_corr(frame: pd.DataFrame, col: str, target: str, max_lag: int) -> pd.Series:
    out = {}
    for lag in range(max_lag + 1):
        pair = pd.concat([frame[col].shift(lag), frame[target]], axis=1).dropna()
        out[lag] = pair.corr(method="spearman").iloc[0, 1]
    return pd.Series(out, name=col)


input_cols = [c for c in num_cols if c not in (target, "% Iron Concentrate")]
lags = pd.concat([lagged_corr(hourly, c, target, max_lag=12) for c in input_cols], axis=1)

best = pd.DataFrame({
    "best_lag_h": lags.abs().idxmax(),
    "corr_at_best": [lags.loc[lags[c].abs().idxmax(), c] for c in lags.columns],
    "corr_lag0": lags.loc[0],
})
print(best.sort_values("corr_at_best", key=abs, ascending=False).round(3).to_string())

lags.plot(figsize=(14, 6), legend=False, xlabel="lag (hours)", ylabel="Spearman corr. with target")

In [ ]:
# Target correlation

acf = pd.Series({lag: hourly[target].autocorr(lag) for lag in range(1, 25)}, name="autocorr")
print(acf.round(3).to_string())

acf.plot(kind="bar", figsize=(10, 4), xlabel="lag (hours)", ylabel="autocorrelation")

In [ ]:
# Amina hypotesis

amina_follows = lagged_corr(hourly, target, "Amina Flow", max_lag=6)
print(amina_follows.round(3).to_string())

---
### Findings: multivariate analysis

Analysis at hourly resolution (inputs averaged per hour, interpolated target hours
excluded, n = 3,787), using Spearman correlation.

**Individually, inputs explain little.** Apart from `% Iron Concentrate`, no input
exceeds |ρ| = 0.21 with the target. The signal, if any, lies in combinations,
interactions and temporal features rather than in a single variable.

**`% Iron Concentrate` is not a feature at lag 0.** It comes from the same lab
sample as the target (ρ = −0.74) and is equally unknown at prediction time. Only its
past values are legitimate inputs.

**Amine dosing responds to silica (closed loop).** In reverse cationic flotation,
more amine should *lower* silica, yet the correlation is positive (+0.21). Shifting
the target instead of the input, the correlation rises to a peak of **0.33 when
silica leads amine by 3 hours**, then decays: operators raise amine after a high lab
result. Correlation here does not indicate causal direction. It also suggests that
sampling, lab analysis and operator reaction take about 3 hours, so a lab result is
not available at its sample timestamp. Lab latency becomes an explicit assumption
in section 1.6.

**No clear residence-time lag.** Lagged correlations (0–12 h) are nearly flat for
all inputs, so lags cannot be chosen from the EDA. Features will use several window
sizes (1, 2, 4, 8 h) and let validation decide.

**Multicollinearity.** `% Iron Feed` and `% Silica Feed` are almost perfectly
inversely correlated (ρ ≈ −1). Levels form two blocks (columns 01–03 and 04–07), and
air flows form a 01–03 block and a 06–07 pair, suggesting two banks of columns.
Bank-level aggregates can replace redundant variables.

**Target autocorrelation defines where the model adds value.**

| Lag (h) | 1 | 2 | 3 | 6 | 12 | 24 |
|---|---|---|---|---|---|---|
| ρ | 0.69 | 0.58 | 0.49 | 0.29 | 0.17 | 0.17 |

At 1–2 hours ahead, the last lab result is a strong predictor (persistence baseline).
Beyond ~6 hours, the target's own past explains little, which is where process
inputs can add value. Repeated lab assays (section 1.3) inflate these values slightly.
---

---
### __1.6 Problem definition__
---

In [ ]:
#Validate Label(remove repeated sample)

y = hourly[target].copy()

run_id = y.ne(y.shift()).cumsum()
run_len = y.groupby(run_id).transform("size")

print(run_len[y.notna()].groupby(run_id[y.notna()]).first().value_counts().sort_index().head(10))

STALE_MIN_HOURS = 4
stale = y.notna() & run_len.ge(STALE_MIN_HOURS)
y_valid = y.mask(stale)

print(f"stale hours: {stale.sum()} in {run_id[stale].nunique()} runs")
print(f"valid labels: {y_valid.notna().sum()} of {y.notna().sum()}")

In [ ]:
# Baseline by horizon

LAB_LATENCY_H = 3


def baseline_scores(y_valid: pd.Series, latency: int, horizons: range) -> pd.DataFrame:
    rows = []
    known = y_valid.ffill(limit=24)
    for h in horizons:
        gap = latency + h
        persistence = known.shift(gap)
        expanding_mean = y_valid.expanding(min_periods=24).mean().shift(gap)
        for name, pred in (("persistence", persistence), ("expanding_mean", expanding_mean)):
            err = (y_valid - pred).dropna()
            rows.append({
                "horizon_h": h,
                "gap_h": gap,
                "baseline": name,
                "n": len(err),
                "MAE": err.abs().mean(),
                "RMSE": np.sqrt((err**2).mean()),
            })
    return pd.DataFrame(rows)


scores = baseline_scores(y_valid, LAB_LATENCY_H, range(0, 7))
print(scores.pivot(index=["horizon_h", "gap_h"], columns="baseline", values="MAE").round(3).to_string())

## 1.6 Problem definition

The decisions below close Stage 1. The full rationale is recorded in
[ADR 0001 — Problem definition](../docs/adr/0001-problem-definition.md).

| # | Decision | Value |
|---|---|---|
| 1 | Valid labels | Hourly lab silica, excluding 310 interpolated hours and 361 hours with a result repeated for ≥ 4 consecutive hours (26 blocks) → **3,426 valid labels** |
| 2 | Lab latency | **L = 2 h**, configurable assumption (not identifiable from the data) |
| 3 | Horizon | h ∈ {0, 1, 2, 4}; **headline h = 2** (4 h ahead of the lab result) |
| 4 | Metric | **MAE (pp)** primary, RMSE secondary; skill = 1 − MAE_model / min(MAE_persistence, MAE_mean) on the same hours |
| 5 | Baselines and success | Persistence and expanding mean (ARIMAX in Stage 4); **success: skill ≥ 0.10 at h = 2** (MAE ≤ ~0.75 pp in walk-forward) |

**Next:** Stage 2 — bronze/silver/gold ingestion in Databricks.